In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataMesh_Aula") \
    .getOrCreate()

print("Spark iniciado!")

Spark iniciado!


In [3]:
clientes = [
    (1, "João", "SP", 35),
    (2, "Maria", "RJ", 28),
    (3, "Carlos", "MG", 42),
    (4, "Ana", "SP", 31),
    (5, "Pedro", "PR", 25)
]

df_clientes = spark.createDataFrame(
    clientes,
    ["cliente_id", "nome", "estado", "idade"]
)

df_clientes.show()

+----------+------+------+-----+
|cliente_id|  nome|estado|idade|
+----------+------+------+-----+
|         1|  João|    SP|   35|
|         2| Maria|    RJ|   28|
|         3|Carlos|    MG|   42|
|         4|   Ana|    SP|   31|
|         5| Pedro|    PR|   25|
+----------+------+------+-----+



In [4]:
vendas = [
    (1001, 1, 3500),
    (1002, 2, 1500),
    (1003, 1, 500),
    (1004, 3, 800),
    (1005, 4, 4200),
    (1006, 5, 1200),
    (1007, 2, 900)
]

df_vendas = spark.createDataFrame(
    vendas,
    ["pedido_id", "cliente_id", "valor"]
)

df_vendas.show()

+---------+----------+-----+
|pedido_id|cliente_id|valor|
+---------+----------+-----+
|     1001|         1| 3500|
|     1002|         2| 1500|
|     1003|         1|  500|
|     1004|         3|  800|
|     1005|         4| 4200|
|     1006|         5| 1200|
|     1007|         2|  900|
+---------+----------+-----+



In [5]:
produtos = [
    (101, "Notebook", "Eletrônicos", 4500),
    (102, "Mouse", "Eletrônicos", 120),
    (103, "Teclado", "Eletrônicos", 250),
    (104, "Cadeira", "Móveis", 900),
    (105, "Monitor", "Eletrônicos", 1500)
]

df_produtos = spark.createDataFrame(
    produtos,
    ["produto_id", "produto", "categoria", "preco"]
)

df_produtos.show()

+----------+--------+-----------+-----+
|produto_id| produto|  categoria|preco|
+----------+--------+-----------+-----+
|       101|Notebook|Eletrônicos| 4500|
|       102|   Mouse|Eletrônicos|  120|
|       103| Teclado|Eletrônicos|  250|
|       104| Cadeira|     Móveis|  900|
|       105| Monitor|Eletrônicos| 1500|
+----------+--------+-----------+-----+



In [6]:
pagamentos = [
    (501, 1001, "APROVADO"),
    (502, 1002, "APROVADO"),
    (503, 1003, "NEGADO"),
    (504, 1004, "APROVADO"),
    (505, 1005, "APROVADO"),
    (506, 1006, "NEGADO"),
    (507, 1007, "APROVADO")
]

df_pagamentos = spark.createDataFrame(
    pagamentos,
    ["pagamento_id", "pedido_id", "status"]
)

df_pagamentos.show()

+------------+---------+--------+
|pagamento_id|pedido_id|  status|
+------------+---------+--------+
|         501|     1001|APROVADO|
|         502|     1002|APROVADO|
|         503|     1003|  NEGADO|
|         504|     1004|APROVADO|
|         505|     1005|APROVADO|
|         506|     1006|  NEGADO|
|         507|     1007|APROVADO|
+------------+---------+--------+



In [7]:
atendimento = [
    (10001, 1, "Problema na entrega"),
    (10002, 2, "Produto com defeito"),
    (10003, 1, "Atraso na entrega"),
    (10004, 3, "Problema no pagamento"),
    (10005, 5, "Cancelamento do pedido"),
    (10006, 2, "Problema na entrega")
]

df_atendimento = spark.createDataFrame(
    atendimento,
    ["chamado_id", "cliente_id", "reclamacao"]
)

df_atendimento.show()

+----------+----------+--------------------+
|chamado_id|cliente_id|          reclamacao|
+----------+----------+--------------------+
|     10001|         1| Problema na entrega|
|     10002|         2| Produto com defeito|
|     10003|         1|   Atraso na entrega|
|     10004|         3|Problema no pagam...|
|     10005|         5|Cancelamento do p...|
|     10006|         2| Problema na entrega|
+----------+----------+--------------------+



In [8]:
df_clientes.select(
    "cliente_id",
    "nome",
    "estado",
    "idade"
).show()

+----------+------+------+-----+
|cliente_id|  nome|estado|idade|
+----------+------+------+-----+
|         1|  João|    SP|   35|
|         2| Maria|    RJ|   28|
|         3|Carlos|    MG|   42|
|         4|   Ana|    SP|   31|
|         5| Pedro|    PR|   25|
+----------+------+------+-----+



In [9]:
from pyspark.sql.functions import sum

df_total_vendas = df_vendas.groupBy(
    "cliente_id"
).agg(
    sum("valor").alias("total_compras")
)

df_total_vendas.show()

+----------+-------------+
|cliente_id|total_compras|
+----------+-------------+
|         1|         4000|
|         2|         2400|
|         5|         1200|
|         3|          800|
|         4|         4200|
+----------+-------------+



In [10]:
from pyspark.sql.functions import count

df_chamados = df_atendimento.groupBy(
    "cliente_id"
).agg(
    count("chamado_id").alias("quantidade_chamados")
)

df_chamados.show()

+----------+-------------------+
|cliente_id|quantidade_chamados|
+----------+-------------------+
|         1|                  2|
|         2|                  2|
|         5|                  1|
|         3|                  1|
+----------+-------------------+



In [11]:
from pyspark.sql.functions import sum, when

df_pagamentos_cliente = df_pagamentos \
    .join(
        df_vendas,
        "pedido_id"
    ) \
    .groupBy("cliente_id") \
    .agg(
        sum(
            when(df_pagamentos.status == "NEGADO", 1)
            .otherwise(0)
        ).alias("pagamentos_negados")
    )

df_pagamentos_cliente.show()

+----------+------------------+
|cliente_id|pagamentos_negados|
+----------+------------------+
|         5|                 1|
|         1|                 1|
|         3|                 0|
|         2|                 0|
|         4|                 0|
+----------+------------------+



In [12]:
df_analytics = df_clientes \
    .join(
        df_total_vendas,
        "cliente_id",
        "left"
    ) \
    .join(
        df_chamados,
        "cliente_id",
        "left"
    ) \
    .join(
        df_pagamentos_cliente,
        "cliente_id",
        "left"
    )

df_analytics.show()

+----------+------+------+-----+-------------+-------------------+------------------+
|cliente_id|  nome|estado|idade|total_compras|quantidade_chamados|pagamentos_negados|
+----------+------+------+-----+-------------+-------------------+------------------+
|         1|  João|    SP|   35|         4000|                  2|                 1|
|         2| Maria|    RJ|   28|         2400|                  2|                 0|
|         5| Pedro|    PR|   25|         1200|                  1|                 1|
|         3|Carlos|    MG|   42|          800|                  1|                 0|
|         4|   Ana|    SP|   31|         4200|               NULL|                 0|
+----------+------+------+-----+-------------+-------------------+------------------+



In [13]:
df_analytics = df_analytics.fillna({
    "total_compras": 0,
    "quantidade_chamados": 0,
    "pagamentos_negados": 0
})

df_analytics.show()

+----------+------+------+-----+-------------+-------------------+------------------+
|cliente_id|  nome|estado|idade|total_compras|quantidade_chamados|pagamentos_negados|
+----------+------+------+-----+-------------+-------------------+------------------+
|         1|  João|    SP|   35|         4000|                  2|                 1|
|         2| Maria|    RJ|   28|         2400|                  2|                 0|
|         5| Pedro|    PR|   25|         1200|                  1|                 1|
|         3|Carlos|    MG|   42|          800|                  1|                 0|
|         4|   Ana|    SP|   31|         4200|                  0|                 0|
+----------+------+------+-----+-------------+-------------------+------------------+



In [14]:
from pyspark.sql.functions import when

df_analytics = df_analytics.withColumn(
    "risco_churn",
    when(
        (df_analytics["quantidade_chamados"] >= 2) |
        (df_analytics["pagamentos_negados"] >= 1),
        "ALTO"
    ).otherwise("BAIXO")
)

df_analytics.show()

+----------+------+------+-----+-------------+-------------------+------------------+-----------+
|cliente_id|  nome|estado|idade|total_compras|quantidade_chamados|pagamentos_negados|risco_churn|
+----------+------+------+-----+-------------+-------------------+------------------+-----------+
|         1|  João|    SP|   35|         4000|                  2|                 1|       ALTO|
|         2| Maria|    RJ|   28|         2400|                  2|                 0|       ALTO|
|         5| Pedro|    PR|   25|         1200|                  1|                 1|       ALTO|
|         3|Carlos|    MG|   42|          800|                  1|                 0|      BAIXO|
|         4|   Ana|    SP|   31|         4200|                  0|                 0|      BAIXO|
+----------+------+------+-----+-------------+-------------------+------------------+-----------+

